# TerraCast · LightGBM Yield Prediction + MLflow 3

Corn / Soybeans 분리 모델 with:
- Yield detrending (state-level linear trend)
- State / County fixed effects
- LightGBM (early stopping)
- MLflow 3 experiment tracking (LoggedModel, Dataset linking)
- 5-Fold CV anomaly detection

**Run in Databricks** (Spark available) or **local Jupyter** (`CSV_PATH` 설정).

In [ ]:
%pip install -U 'mlflow>=3.1' lightgbm geopandas

In [ ]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold, train_test_split
from numpy.polynomial import polynomial as P

import mlflow
import mlflow.lightgbm
from mlflow.entities import Dataset

# Spark session in Databricks; None locally
try:
    spark = globals().get("spark", None)
except NameError:
    spark = None

print("MLflow version:", mlflow.__version__)

## Config

In [ ]:
# --- Edit these ---
MLFLOW_EXPERIMENT_ID = "2445807330421530"   # Databricks MLflow experiment ID
MERGED_TABLE         = "workspace.default.merged"  # Spark table (Databricks)
CSV_PATH             = None                 # local CSV fallback, e.g. Path("/path/to/merged.csv")

# LightGBM hyperparameters
LGBM_PARAMS = dict(
    n_estimators      = 1000,
    learning_rate     = 0.05,
    num_leaves        = 63,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.6,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = 42,
    verbose           = -1,
)

N_CV_FOLDS   = 5
ANOMALY_Z    = 2.0
RANDOM_STATE = 42

## Data Loading

In [ ]:
if spark is not None:
    df_raw = spark.table(MERGED_TABLE).toPandas()
elif CSV_PATH and Path(CSV_PATH).exists():
    df_raw = pd.read_csv(CSV_PATH)
else:
    raise ValueError("Set spark + MERGED_TABLE, or a valid CSV_PATH")

df_raw.columns = [c.strip() for c in df_raw.columns]
df_raw["commodity"] = df_raw["commodity"].str.strip()
print(f"Loaded rows: {len(df_raw)}")

## Preprocessing & Feature Engineering

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing values and engineer all features. Returns a new DataFrame."""
    df = df.copy()

    # ── Numeric cast (Spark/CSV에서 object로 로드될 수 있음) ──
    numeric_cols = [
        "avg_temp_max_c", "avg_temp_min_c", "avg_temp_mean_c", "peak_temp_max_c",
        "winter_snowfall_mm", "avg_snow_depth_mm", "max_snow_depth_mm", "snow_days",
        "avg_precip_mm", "precip_std_mm", "heat_stress_days", "gdd",
        "dry_days", "heavy_rain_days", "frost_days",
        "drought_flag", "flood_flag", "extreme_heat_flag",
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # ── Imputation ────────────────────────────────────────────
    temp_cols = ["avg_temp_max_c", "avg_temp_min_c", "avg_temp_mean_c", "peak_temp_max_c"]
    for c in temp_cols:
        df[c] = df[c].fillna(
            df.groupby(["commodity", "state_abbr"])[c].transform("median")
        )
    for c in ["winter_snowfall_mm", "avg_snow_depth_mm", "max_snow_depth_mm", "snow_days"]:
        df[c] = df[c].fillna(0)
    df["avg_precip_mm"]  = df["avg_precip_mm"].fillna(df["avg_precip_mm"].median())
    df["precip_std_mm"]  = df["precip_std_mm"].fillna(df["precip_std_mm"].median())

    # ── Feature engineering ───────────────────────────────────
    df["is_irrigated"]       = (df["irrigation"].str.strip() == "Irrigated").astype(int)
    df["temp_range"]         = df["avg_temp_max_c"] - df["avg_temp_min_c"]
    df["peak_vs_mean_temp"]  = df["peak_temp_max_c"] - df["avg_temp_mean_c"]
    df["temp_stress_index"]  = df["peak_temp_max_c"] * df["heat_stress_days"]
    df["precip_cv"]          = df["precip_std_mm"] / (df["avg_precip_mm"] + 1)
    df["heat_dry_combo"]     = df["heat_stress_days"] * df["dry_days"]
    df["extreme_events"]     = df["drought_flag"] + df["extreme_heat_flag"]
    df["gdd_precip"]         = df["gdd"] * df["avg_precip_mm"]
    df["gdd_per_precip"]     = df["gdd"] / (df["avg_precip_mm"] + 1)
    df["year_trend"]         = df["year"] - 2010
    df["peak_temp_sq"]       = df["peak_temp_max_c"] ** 2
    df["log_heat_days"]      = np.log1p(df["heat_stress_days"])
    df["optimal_temp_score"] = -abs(df["avg_temp_mean_c"] - 22.5)
    df["temprange_x_heat"]   = df["temp_range"] * df["heat_stress_days"]
    df["peak_x_heat"]        = df["peak_temp_max_c"] * df["heat_stress_days"]
    df["extreme_x_precip"]   = df["extreme_heat_flag"] * df["avg_precip_mm"]
    df["gdd_x_heat"]         = df["gdd"] * df["heat_stress_days"]
    df["gdd_x_precip"]       = df["gdd"] * df["avg_precip_mm"]
    df["precip_x_snow"]      = df["avg_precip_mm"] * df["snow_days"]
    df["extreme_x_dry"]      = df["extreme_events"] * df["dry_days"]
    df["gdd_x_extreme"]      = df["gdd"] * df["extreme_events"]
    df["stress_gdd_ratio"]   = df["heat_stress_days"] * df["peak_temp_max_c"] / (df["gdd"] + 1)
    df["cool_moist"]         = (50 - df["avg_temp_min_c"]) * df["avg_precip_mm"]
    df["gdd_precip_heat"]    = df["gdd"] * df["avg_precip_mm"] / (df["heat_stress_days"] + 1)
    yt = df["year_trend"]
    df["year_x_extreme"]     = yt * df["extreme_events"]
    df["year_x_precip"]      = yt * df["avg_precip_mm"]
    df["year_x_heat"]        = yt * df["heat_stress_days"]

    return df


WEATHER_FEATURES = [
    "avg_temp_max_c", "avg_temp_mean_c", "peak_temp_max_c",
    "temp_range", "peak_vs_mean_temp", "temp_stress_index", "heat_stress_days",
    "peak_temp_sq", "log_heat_days", "optimal_temp_score",
    "avg_precip_mm", "dry_days", "heavy_rain_days", "precip_cv",
    "heat_dry_combo", "extreme_events", "drought_flag", "extreme_heat_flag",
    "extreme_x_dry", "gdd_x_extreme", "stress_gdd_ratio",
    "gdd", "gdd_precip", "gdd_per_precip", "gdd_precip_heat", "cool_moist",
    "frost_days", "winter_snowfall_mm", "snow_days",
    "temprange_x_heat", "peak_x_heat", "extreme_x_precip",
    "gdd_x_heat", "gdd_x_precip", "precip_x_snow",
    "year_trend", "year_x_extreme", "year_x_precip", "year_x_heat",
    "is_irrigated",
]

df = build_features(df_raw)
print(f"Features ready. Shape: {df.shape}")

## Helper functions

In [ ]:
def compute_metrics(actual, predicted):
    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    mae  = float(mean_absolute_error(actual, predicted))
    r2   = float(r2_score(actual, predicted))
    return rmse, mae, r2


def add_detrend_and_fixed_effects(train: pd.DataFrame, test: pd.DataFrame):
    """Add trend, yield_anomaly, state_baseline, county_baseline columns."""
    # State-level linear detrend (fit on train only)
    trend_params = {}
    for state, grp in train.groupby("state_abbr"):
        if len(grp) >= 3:
            coeffs = P.polyfit(grp["year"].values.astype(float),
                               grp["yield_amount"].values, deg=1)
        else:
            coeffs = np.array([grp["yield_amount"].mean(), 0.0])
        trend_params[state] = coeffs
    global_coeffs = P.polyfit(
        train["year"].values.astype(float), train["yield_amount"].values, deg=1
    )

    for part in [train, test]:
        part["trend"] = np.nan
        for state, coeffs in trend_params.items():
            m = part["state_abbr"] == state
            if m.any():
                part.loc[m, "trend"] = P.polyval(
                    part.loc[m, "year"].values.astype(float), coeffs
                )
        na = part["trend"].isna()
        if na.any():
            part.loc[na, "trend"] = P.polyval(
                part.loc[na, "year"].values.astype(float), global_coeffs
            )
        part["yield_anomaly"] = part["yield_amount"] - part["trend"]

    # Fixed effects (computed from train)
    state_mean = train.groupby("state_abbr")["yield_anomaly"].mean()
    train["state_baseline"] = train["state_abbr"].map(state_mean).fillna(0)
    test["state_baseline"]  = test["state_abbr"].map(state_mean).fillna(0)

    cmean = (
        train.groupby(["state_fips", "county_fips"])["yield_anomaly"]
        .mean().reset_index(name="county_baseline")
    )
    train = train.merge(cmean, on=["state_fips", "county_fips"], how="left")
    test  = test.merge(cmean, on=["state_fips", "county_fips"], how="left")
    train["county_baseline"] = train["county_baseline"].fillna(0)
    test["county_baseline"]  = test["county_baseline"].fillna(0)

    return train, test

## MLflow Training — Corn & Soybeans (8:2 split)

In [ ]:
mlflow.set_experiment(experiment_id=MLFLOW_EXPERIMENT_ID)

# County-wise 8:2 train/test split
county_ids = df[["state_fips", "county_fips"]].drop_duplicates()
np.random.seed(RANDOM_STATE)
test_idx = np.random.choice(
    len(county_ids), size=int(len(county_ids) * 0.2), replace=False
)
test_set = set(map(tuple, county_ids.iloc[test_idx].values))
df["_key"]    = list(zip(df["state_fips"], df["county_fips"]))
df["is_test"] = df["_key"].isin(test_set)
df.drop(columns="_key", inplace=True)

FEATURES = WEATHER_FEATURES + ["state_baseline", "county_baseline"]
summary  = {}

with mlflow.start_run(run_name="lgbm_yield_prediction") as parent_run:
    mlflow.log_params({f"lgbm_{k}": v for k, v in LGBM_PARAMS.items()})

    for crop in ["Corn", "Soybeans"]:
        with mlflow.start_run(run_name=f"lgbm_{crop.lower()}", nested=True) as child_run:
            dc    = df[df["commodity"] == crop].copy()
            train = dc[~dc["is_test"]].copy()
            test  = dc[dc["is_test"]].copy()
            print(f"\n{'='*55}\n  {crop}  Train={len(train)}, Test={len(test)}\n{'='*55}")

            train, test = add_detrend_and_fixed_effects(train, test)

            X_train_all = train[FEATURES]
            y_train_all = train["yield_anomaly"]
            X_test_fin  = test[FEATURES]
            y_test_fin  = test["yield_anomaly"]

            # Register datasets with MLflow
            train_dataset: Dataset = mlflow.data.from_pandas(
                train[["state_fips", "county_fips", "year"] + FEATURES + ["yield_anomaly"]],
                name=f"train_{crop.lower()}",
            )
            test_dataset: Dataset = mlflow.data.from_pandas(
                test[["state_fips", "county_fips", "year"] + FEATURES + ["yield_anomaly"]],
                name=f"test_{crop.lower()}",
            )

            # Early stopping validation split (inside train)
            X_tr, X_val, y_tr, y_val = train_test_split(
                X_train_all, y_train_all, test_size=0.15, random_state=RANDOM_STATE
            )

            model = lgb.LGBMRegressor(**LGBM_PARAMS)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False)],
            )
            print(f"  Best iteration: {model.best_iteration_}")

            # Log model (linked to LoggedModel entity)
            model_info = mlflow.sklearn.log_model(
                sk_model      = model,
                name          = f"lgbm_{crop.lower()}",
                params        = {**LGBM_PARAMS, "best_iteration": model.best_iteration_},
                input_example = X_tr.head(5),
            )
            logged_model = mlflow.get_logged_model(model_info.model_id)
            print(f"  model_id: {logged_model.model_id}")

            # ── Train metrics ────────────────────────────────────
            train_pred_anomaly = model.predict(X_train_all)
            train_pred_yield   = train_pred_anomaly + train["trend"].values
            train_rmse, train_mae, train_r2 = compute_metrics(
                train["yield_amount"].values, train_pred_yield
            )
            mlflow.log_metrics(
                metrics  = {"train_r2": train_r2, "train_mae": train_mae, "train_rmse": train_rmse},
                model_id = logged_model.model_id,
                dataset  = train_dataset,
            )

            # ── Test metrics ─────────────────────────────────────
            pred_anomaly = model.predict(X_test_fin)
            pred_yield   = pred_anomaly + test["trend"].values
            actual_yield = test["yield_amount"].values
            rmse, mae, r2 = compute_metrics(actual_yield, pred_yield)

            mlflow.log_metrics(
                metrics  = {"test_r2": r2, "test_mae": mae, "test_rmse": rmse},
                model_id = logged_model.model_id,
                dataset  = test_dataset,
            )

            print(f"  R2={r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}")
            summary[crop] = {"r2": r2, "mae": mae, "rmse": rmse, "model_id": logged_model.model_id}

    # Log combined summary to parent run
    for crop, m in summary.items():
        mlflow.log_metrics({
            f"{crop.lower()}_r2":   m["r2"],
            f"{crop.lower()}_mae":  m["mae"],
            f"{crop.lower()}_rmse": m["rmse"],
        })

print("\nTraining complete.")
for crop, m in summary.items():
    print(f"  {crop:10s}: R2={m['r2']:.4f}  MAE={m['mae']:.2f}  RMSE={m['rmse']:.2f}  model_id={m['model_id']}")

## 5-Fold CV Anomaly Detection

In [ ]:
county_ids = df[["state_fips", "county_fips"]].drop_duplicates().reset_index(drop=True)
kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

df["pred_yield"] = np.nan

with mlflow.start_run(run_name=f"lgbm_{N_CV_FOLDS}fold_cv_anomaly") as cv_run:
    mlflow.log_params({
        **{f"lgbm_{k}": v for k, v in LGBM_PARAMS.items()},
        "n_cv_folds": N_CV_FOLDS,
        "anomaly_z_threshold": ANOMALY_Z,
    })

    fold_r2s = []

    for fold, (train_county_idx, test_county_idx) in enumerate(kf.split(county_ids)):
        test_counties = set(map(tuple, county_ids.iloc[test_county_idx].values))
        key       = list(zip(df["state_fips"], df["county_fips"]))
        test_mask = np.array([k in test_counties for k in key])

        fold_preds, fold_actuals = [], []

        for crop in ["Corn", "Soybeans"]:
            crop_mask = df["commodity"] == crop
            tr = df[~test_mask & crop_mask].copy()
            te = df[ test_mask & crop_mask].copy()
            if len(te) == 0:
                continue

            tr, te = add_detrend_and_fixed_effects(tr, te)

            FEATURES = WEATHER_FEATURES + ["state_baseline", "county_baseline"]

            model = lgb.LGBMRegressor(**LGBM_PARAMS)
            val_sample = tr[FEATURES].sample(frac=0.15, random_state=RANDOM_STATE)
            model.fit(
                tr[FEATURES], tr["yield_anomaly"],
                eval_set=[(val_sample, tr["yield_anomaly"].loc[val_sample.index])],
                callbacks=[lgb.early_stopping(50, verbose=False)],
            )

            pred_anomaly = model.predict(te[FEATURES])
            pred_yield   = pred_anomaly + te["trend"].values
            df.loc[te.index, "pred_yield"] = pred_yield

            fold_preds.extend(pred_yield.tolist())
            fold_actuals.extend(te["yield_amount"].values.tolist())

        if fold_actuals:
            fold_r2 = r2_score(fold_actuals, fold_preds)
            fold_r2s.append(fold_r2)
            mlflow.log_metric("fold_r2", fold_r2, step=fold + 1)
            print(f"  Fold {fold+1}/{N_CV_FOLDS} done — R2={fold_r2:.4f}")

    # ── Residual → Z-score → Anomaly flag ────────────────────
    df["residual"] = (df["pred_yield"] - df["yield_amount"]).abs()
    df["residual_zscore"] = df.groupby("commodity")["residual"].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    df["anomaly_flag"] = (df["residual_zscore"] > ANOMALY_Z).astype(int)

    total_anomalies = int(df["anomaly_flag"].sum())
    anomaly_pct     = float(df["anomaly_flag"].mean() * 100)
    cv_r2_mean      = float(np.mean(fold_r2s))

    mlflow.log_metrics({
        "cv_r2_mean":      cv_r2_mean,
        "total_anomalies": total_anomalies,
        "anomaly_pct":     anomaly_pct,
    })

    print(f"\nCV R2 mean : {cv_r2_mean:.4f}")
    print(f"Anomalies  : {total_anomalies} ({anomaly_pct:.1f}%)")

## Anomaly Summary

In [ ]:
print("Crop별 anomaly 현황:")
print(df.groupby("commodity")["anomaly_flag"].agg(["sum", "mean", "count"]))

out_cols = [
    "state_fips", "county_fips", "year", "county_name", "state_abbr",
    "commodity", "irrigation", "yield_amount", "pred_yield",
    "residual", "residual_zscore", "anomaly_flag",
]
result_df = df[[c for c in out_cols if c in df.columns]].copy()

display(result_df[result_df["anomaly_flag"] == 1].sort_values("residual_zscore", ascending=False).head(20))